In [1]:
# Cell 1 description and imports
"""
Baseline Generator Notebook

Generates fatigue and actuator usage budgets for two modes:
- 'greedy': uses full time series (or 3D joint distibutions) with normal turbine operation (no WF control)
- 'IEC': uses a joint 2D distribution of wind conditions for a single turbine based on the IEC class definition

Outputs:
- fatigue_budget.json → dictionary with per-metric lifetime budget
- turbine_metrics.parquet → per-turbine summary
- settings.json → metadata

This is the reference for lifetime usage comparison in controlled simulations.
"""
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import warnings
# ignore only those JAX scatter FutureWarnings
warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    message=r"scatter inputs have incompatible types: cannot safely cast value from dtype=float64 to dtype=float32.*"
)

import os
import numpy as np
import pandas as pd
import json
from pathlib import Path
from input_handling import (
     load_wind_timeseries, load_iec_distribution, load_joint_distribution, load_surrogate_models
)
from simulation_engine import simulate_block, simulate_distribution,simulate_distribution_single_turbine
from postprocessing import aggregate_farm_budget, save_results
from control_interface import constant_control,combine_controllers
from farm_setup import initialize_HKN_pywake_farm, compute_effective_inflow,initialize_single_turbine_farm_IEA22



In [2]:
# Cell 2 Parameter definition 

# Baseline mode: choose "greedy_timeseries" or "greedy_distribution" or "IEC"
baseline_mode = "IEC"

# Input files
# wind_price_TS_file = "data/timeseries/HKNB_timeseries_full_filled_small_gaps_only_TI_boost.csv"    # used in greedy timeseries mode
# distribution_file = "distributions/HKNB170m_WSWDTIprobs.csv"  # used in IEC or greedy_distribution mode 
# distribution_file = "distributions/HKNB_Weib_WSWD_TI_dist_3deg_clip_TIboost.csv"  
distribution_file = "distributions/IEC_IC_distribution.csv"  # used in IEC or greedy_distribution mode

# Initialize farm from a function (pywake or floris)
layout = "layout/HKN_layout_subset_with_scaled.csv"
# farm = initialize_HKN_pywake_farm(layout) # load pywake or floris farm setup
farm = initialize_single_turbine_farm_IEA22() # load single turbine farm setup for IEC mode

# Output
baseline_name = "IEC_IC_baseline_IEA22MW"  # Name of the baseline for output files
#  Flags to save time series olnly relevant in timeseries mode
return_turbine_timeseries=True # If True, save turbine-level time series in the output parquet file
return_turbine_timeseries_cum=True # If True, save cumulative turbine-level time series in the output parquet file
return_farm_timeseries=True # If True, save farm-level time series in the output parquet file
return_bin_outputs=True  # If True, save bin outputs (e.g., actuator counts) in the output parquet file

# Input resolution: "10min" or "1h"
input_resolution = "10min"

# Basis lifetime for damage metrics
lifetime_years = 25  # Needed for distribution modes

# Price input (optional)

use_prices = False # If false, orice not used and the next 2 parameters are ignored
price_type = "DA_spot"  # "DA_spot" or "fixed_PPA"
fixed_price_value = 80.0  # EUR/MWh, used if price_type == "fixed_PPA"

# Wind speed operational parameters
wsp_cut_in =3 # m/s cut-in wind speed
wsp_cut_off = 25 # m/s cut-off wind speed


# Surrogate options
model_path_base="models/RA_IEA22_DTU" # Path to surrgoate keras models
scaler_path_base="models/RA_IEA22_DTU" # Path to surrogate scalers
use_sector_average = False  # If True, use sector average based surrogate
supports_operating_modes = False # If True, the surrogate supports operating modes besides normal operation (idling, startup, shutdown)

channels = [
    "RA_ADC", "RA_BRM","RA_ebrm", "RA_fbrm", "RA_blade_torsion", "RA_tbfa",
    "RA_tbss", "RA_TBM", "RA_TB_torsion","RA_ttfa", "RA_TTM","RA_gsb_l10","RA_rsb_l10","RA_shaft_mx_mb_fixed","RA_shaft_mz_mb_fixed"
]
channel_processing_specs = {
    # DEL-based damage metrics (DEL → D)
    "RA_tbfa": {"type": "damage", "wohler": 4, "nref": 600},
    "RA_tbss": {"type": "damage", "wohler": 4, "nref": 600},
    "RA_TBM": {"type": "damage", "wohler": 4, "nref": 600},
    "RA_TB_torsion": {"type": "damage", "wohler": 4, "nref": 600},
    "RA_TTM": {"type": "damage", "wohler": 4, "nref": 600},
    "RA_ttfa": {"type": "damage", "wohler": 4, "nref": 600},
    "RA_BRM": {"type": "damage", "wohler": 10, "nref": 600},
    "RA_ebrm": {"type": "damage", "wohler": 10, "nref": 600},
    "RA_fbrm": {"type": "damage", "wohler": 10, "nref": 600},
    "RA_blade_torsion": {"type": "damage", "wohler": 10, "nref": 600},  
    "RA_shaft_mx_mb_fixed": {"type": "damage", "wohler": 4, "nref": 600},
    "RA_shaft_mz_mb_fixed": {"type": "damage", "wohler": 4, "nref": 600},
    # Harmonic mean channels (bearing life rates)
    "RA_gsb_l10": {"type": "harmonic"},
    "RA_rsb_l10": {"type": "harmonic"},

    # Simple accumulators (e.g. actuators, custom metrics)
    "RA_ADC": {"type": "sum"},  # actuator counts (e.g., pitch)
    
    # Handled internally — not in surrogates
    # "YawTravel": {"type": "sum"},
    # "Energy": {"type": "sum"},
    # "Revenue": {"type": "sum"},  # only if prices are used
}


In [3]:
# Cell 3 Main execution cell

# Load surrogate models and metadata
surrogate,surrogate_metadata = load_surrogate_models(
    model_path_base=model_path_base,
    scaler_path_base=scaler_path_base,
    channels=channels,
    use_sector_average=use_sector_average,
    supports_operating_modes=supports_operating_modes
)

# Initialize farm and control functions
layout = farm["layout_df"]
base_control = constant_control(yaw=0, power=100.0, shutdown=False)
control_fn = combine_controllers(base_control, rule_wrappers=[])

if baseline_mode == "greedy_timeseries":
    wind_data = load_wind_timeseries(wind_price_TS_file, input_resolution,require_price=use_prices)
    
    if use_prices and price_type == "DA_spot":
        # Price comes from wind_data file
        if "price" not in wind_data.columns:
            raise ValueError("Expected price column not found in wind timeseries.")
        price_series = wind_data["price"].values
    elif use_prices and price_type == "fixed_PPA":
        price_series = np.full(len(wind_data), fixed_price_value)
    else:
        price_series = None

    result_df,turb_ts_inst,turb_ts_cum,farm_ts = simulate_block(
        wind_data=wind_data,
        farm=farm,
        surrogate=surrogate,
        control_fn=control_fn,
        resolution=input_resolution,
        use_sector_average=use_sector_average,  
        supports_operating_modes=supports_operating_modes,
        channel_processing_specs=channel_processing_specs,
        wsp_min=wsp_cut_in,
        wsp_max=wsp_cut_off,
        price_series=price_series,
        baseline_df=None,       
        aux_df=None, 
        return_turbine_timeseries=return_turbine_timeseries,
        return_turbine_timeseries_cum= return_turbine_timeseries_cum,
        return_farm_timeseries= return_farm_timeseries,
    )
    bin_values = None
elif baseline_mode == "greedy_distribution":
    if use_prices and price_type == "DA_spot":     
        distribution = load_joint_distribution(distribution_file, require_price=use_prices)
        if "price" not in distribution.columns:
            raise ValueError("Distribution must include 'price' column if prices are used.")
        price_type = price_type  # just for clarity
    elif use_prices and price_type == "fixed_PPA":
        distribution = load_joint_distribution(distribution_file, require_price=False)
        distribution["price"] = fixed_price_value
        price_type = price_type  # just for clarity
    else:
        price_type = None
        distribution = load_joint_distribution(distribution_file, require_price=False)

    result_df, bin_df = simulate_distribution(distribution,
                          farm,
                          surrogate,
                          control_fn,
                          channel_processing_specs,
                          lifetime_years = lifetime_years,
                          wsp_min= wsp_cut_in,
                          wsp_max= wsp_cut_off,
                          supports_operating_modes = supports_operating_modes,
                          use_sector_average = use_sector_average,
                          use_prices=use_prices,
                          price_type=price_type,
                          fixed_price_value=fixed_price_value,
                          return_bin_outputs=return_bin_outputs)
    turb_ts_inst = None
    turb_ts_cum = None
    farm_ts = None
    bin_values = bin_df
    

elif baseline_mode == "IEC":
    distribution = load_iec_distribution(distribution_file)
    
    result_df = simulate_distribution_single_turbine(
        distribution,
        surrogate,
        farm,
        channel_processing_specs,
        lifetime_years=lifetime_years,        
        use_sector_average=False,
        wsp_min=3,
        wsp_max=25
    )
    turb_ts_inst = None
    turb_ts_cum = None
    farm_ts = None
    bin_values = None

 
else:
    raise ValueError("Invalid baseline_mode.")

# Global max per channel (greedy) or direct value (IEC)
# Include all load, actuator, power, and revenue metrics
baseline_budget = aggregate_farm_budget(result_df)

include_prices = use_prices and baseline_mode in ["greedy_timeseries", "greedy_distribution"]

settings = {
    "mode": baseline_mode,
    "input_resolution": input_resolution if 'timeseries' in baseline_mode else None,
    "source": wind_price_TS_file if 'timeseries' in baseline_mode else distribution_file,
    "num_turbines": len(layout) if 'greedy' in baseline_mode else 1,
    "lifetime_years": lifetime_years if baseline_mode in ["IEC", "greedy_distribution"] else None,
    "price_source": (
        distribution_file if (baseline_mode == "greedy_distribution" and price_type == "DA_spot")
        else wind_price_TS_file if (baseline_mode == "greedy_timeseries" and price_type == "DA_spot")
        else "fixed_PPA" if (price_type == "fixed_PPA")
        else None
    ),
    "price_type": price_type if include_prices else None,
    "fixed_price": fixed_price_value if (include_prices and price_type == "fixed_PPA") else None,
    "control_settings": control_fn.describe() if hasattr(control_fn, 'describe') else {"type": "unknown"},
    "wsp_cut_in": wsp_cut_in,
    "wsp_cut_out": wsp_cut_off,
    "flow_type": farm.get("flow_type"),
    "farm_setup": farm.get("setup_summary"),
    "rotor_model": farm.get("rotor_model"),
    "num_turbines": int(farm["layout_df"].shape[0]),
    "surrogate": surrogate_metadata
    }

# Path setup for output
output_path = Path(f"baselines/{baseline_name}")
output_path.mkdir(parents=True, exist_ok=True)

save_results(
    output_path=output_path,
    turbine_metrics=result_df,
    fatigue_budget=baseline_budget,
    settings=settings,
    farm_metrics=None,
    comparison_to_baseline=None,
    turb_ts_inst =turb_ts_inst,
    turb_ts_cum =turb_ts_cum,
    farm_ts=farm_ts,
    bin_values=bin_values
    )


E0000 00:00:1756464880.229500  105530 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1756464880.245629  105530 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1756464880.345742  105530 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1756464880.345785  105530 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1756464880.345788  105530 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1756464880.345790  105530 computation_placer.cc:177] computation placer already registered. Please check linka

In [4]:
# inflow_df.to_parquet("inflow_df_base.parquet", index=False)
print(inflow_df.head())


NameError: name 'inflow_df' is not defined

In [ ]:
# import pandas as pd
# import numpy as np

# # --- 1. Load CSV
# ambient_df = pd.read_csv("/groups/SUDOCO/Task23/sudoco_task2.3/data/timeseries/HKNB_timeseries_full_filled_small_gaps_only.csv")  # <-- or replace with your existing DataFrame
# print(f"Loaded ambient data with {len(ambient_df)} rows")

# # --- 2. Construct surrogate input matrix
# wsp_series = ambient_df["wsp"].values
# TI_series = ambient_df["TI"].values * 100.0  # Convert to percent
# yaw_series = np.zeros_like(wsp_series)
# power_series = np.full_like(wsp_series, 100.0)

# TI_series = np.clip(TI_series, 4.5, 34.9)
# X_test = np.stack([wsp_series, TI_series, yaw_series, power_series], axis=1)

# # --- 3. Get surrogate model
# some_channel = list(surrogate.keys())[0]
# model = surrogate[some_channel]

# # --- 4. In-domain mask
# mask = model.domain.in_domain(X_test)
# print(f"\n {np.sum(mask)} in-domain")
# print(f" {np.sum(~mask)} out-of-domain")

# # --- 5. Bounds
# min_bounds = model.domain.min
# max_bounds = model.domain.max
# input_names = ["WS", "TI", "yaw", "power"]

# # --- 6. Collect offending rows
# rows_out = []
# for idx, row in zip(ambient_df.index[~mask], X_test[~mask]):
#     violations = {}
#     for j in range(row.shape[0]):
#         if row[j] < min_bounds[j] or row[j] > max_bounds[j]:
#             violations[input_names[j]] = f"{row[j]:.2f} (bounds: [{min_bounds[j]:.2f}, {max_bounds[j]:.2f}])"
#     rows_out.append({
#         "index": idx,
#         "wsp": row[0],
#         "TI": row[1],
#         "yaw": row[2],
#         "power": row[3],
#         "violations": violations
#     })

# # --- 7. Save to variable for manual inspection
# df_out_of_bounds = pd.DataFrame(rows_out)
